# Hyperparameter Tuning System
### Automated Model Optimization — Grid Search & Random Search
---
This notebook:
- Loads & preprocesses your dataset
- Trains baseline models (Random Forest, XGBoost, Gradient Boosting, SVM)
- Runs GridSearchCV and RandomizedSearchCV
- Visualises parameter impact & overfitting
- Generates an auto leaderboard and final recommendation report


## 1. Imports & Configuration

In [ ]:
import warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, cross_val_score
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13, 'axes.labelsize': 11})

print('All libraries loaded successfully.')


## 2. Dataset Loading

In [ ]:
# UPDATE THIS PATH to your uploaded dataset
filepath = 'heart.csv'

df = pd.read_csv(filepath)

print('=' * 60)
print(f'  Dataset shape : {df.shape[0]} rows x {df.shape[1]} columns')
print('=' * 60)
print('\nColumn dtypes:')
print(df.dtypes.to_string())
print('\nFirst 5 rows:')
display(df.head())
print('\nBasic statistics:')
display(df.describe(include='all'))
print('\nMissing values per column:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else '  -> No missing values found.')


## 3. Data Preprocessing

In [ ]:
df_proc = df.copy()

# Identify target (last column)
target_col = df_proc.columns[-1]
print(f'Target column : "{target_col}"')
print(f'Classes       : {sorted(df_proc[target_col].unique())}')

num_cols = df_proc.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_proc.select_dtypes(include=['object', 'category']).columns.tolist()
if target_col in num_cols: num_cols.remove(target_col)
if target_col in cat_cols: cat_cols.remove(target_col)

# Handle missing values
if df_proc[num_cols].isnull().any().any():
    imp_num = SimpleImputer(strategy='median')
    df_proc[num_cols] = imp_num.fit_transform(df_proc[num_cols])
    print('Numeric missing values -> imputed with median')

if cat_cols and df_proc[cat_cols].isnull().any().any():
    imp_cat = SimpleImputer(strategy='most_frequent')
    df_proc[cat_cols] = imp_cat.fit_transform(df_proc[cat_cols])
    print('Categorical missing values -> imputed with mode')

# Encode categoricals
if cat_cols:
    df_proc = pd.get_dummies(df_proc, columns=cat_cols, drop_first=False)
    print(f'One-hot encoded columns: {cat_cols}')
    print(f'New shape after encoding: {df_proc.shape}')
else:
    print('No categorical columns to encode.')

# Encode target if needed
if df_proc[target_col].dtype == object:
    le = LabelEncoder()
    df_proc[target_col] = le.fit_transform(df_proc[target_col])
    print(f'Target label-encoded: {dict(zip(le.classes_, le.transform(le.classes_)))}')

X = df_proc.drop(columns=[target_col])
y = df_proc[target_col]

# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'\nTrain size : {X_train.shape[0]}  |  Test size : {X_test.shape[0]}')
print(f'Features   : {X_train.shape[1]}')
print('\nPreprocessing complete.')


## 4. Baseline Models (Default Parameters)

In [ ]:
models = {
    'Random Forest'     : RandomForestClassifier(random_state=RANDOM_STATE),
    'XGBoost'           : XGBClassifier(random_state=RANDOM_STATE,
                                        use_label_encoder=False,
                                        eval_metric='logloss', verbosity=0),
    'Gradient Boosting' : GradientBoostingClassifier(random_state=RANDOM_STATE),
    'SVM'               : SVC(random_state=RANDOM_STATE, probability=True)
}

baseline_results = {}

print('=' * 55)
print('  BASELINE MODEL PERFORMANCE (Default Parameters)')
print('=' * 55)

for name, model in models.items():
    X_tr = X_train_sc if name == 'SVM' else X_train
    X_te = X_test_sc  if name == 'SVM' else X_test

    t0 = time.time()
    model.fit(X_tr, y_train)
    elapsed = time.time() - t0

    train_acc = accuracy_score(y_train, model.predict(X_tr))
    test_acc  = accuracy_score(y_test,  model.predict(X_te))

    baseline_results[name] = {
        'model': model, 'train_acc': train_acc,
        'test_acc': test_acc, 'time': elapsed
    }

    print(f'  {name:<22} Default Accuracy = {test_acc*100:.1f}%  '
          f'(train={train_acc*100:.1f}%)  [{elapsed:.1f}s]')

print('=' * 55)


## 5. Parameter Grids

In [ ]:
param_grids = {
    'Random Forest': {
        'n_estimators'     : [100, 200, 300],
        'max_depth'        : [5, 10, 15],
        'min_samples_split': [2, 4, 6]
    },
    'XGBoost': {
        'n_estimators' : [100, 200, 300],
        'max_depth'    : [3, 6, 9],
        'learning_rate': [0.01, 0.1, 0.2]
    },
    'Gradient Boosting': {
        'n_estimators' : [100, 200, 300],
        'max_depth'    : [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2]
    },
    'SVM': {
        'C'     : [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'gamma' : ['scale', 'auto']
    }
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print('Parameter grids defined.')
print('Cross-validation: StratifiedKFold (n_splits=5)')
for name, grid in param_grids.items():
    n_combos = 1
    for v in grid.values(): n_combos *= len(v)
    print(f'  {name:<22}: {n_combos} combinations')


## 6. Grid Search (GridSearchCV)

In [ ]:
grid_results = {}

print('=' * 60)
print('  GRID SEARCH RESULTS')
print('=' * 60)

for name, model in models.items():
    X_tr = X_train_sc if name == 'SVM' else X_train
    X_te = X_test_sc  if name == 'SVM' else X_test

    fresh_model = model.__class__(**{
        k: v for k, v in model.get_params().items()
        if k in model.__class__().get_params()
    })

    gs = GridSearchCV(
        fresh_model, param_grids[name],
        cv=cv, scoring='accuracy', n_jobs=-1, verbose=0
    )

    t0 = time.time()
    gs.fit(X_tr, y_train)
    elapsed = time.time() - t0

    test_acc  = accuracy_score(y_test,  gs.predict(X_te))
    train_acc = accuracy_score(y_train, gs.predict(X_tr))

    grid_results[name] = {
        'best_estimator': gs.best_estimator_,
        'best_params'   : gs.best_params_,
        'cv_score'      : gs.best_score_,
        'test_acc'      : test_acc,
        'train_acc'     : train_acc,
        'time'          : elapsed,
        'gs_object'     : gs
    }

    print(f'\n  {name}')
    print(f'    Best CV Accuracy : {gs.best_score_*100:.2f}%')
    print(f'    Test  Accuracy   : {test_acc*100:.2f}%')
    print(f'    Training Time    : {elapsed:.1f}s')
    print(f'    Best Parameters  : {gs.best_params_}')

print('\n' + '=' * 60)
print('Grid search complete for all models.')


## 7. Random Search (RandomizedSearchCV)

In [ ]:
random_results = {}

print('=' * 60)
print('  RANDOM SEARCH RESULTS')
print('=' * 60)

for name, model in models.items():
    X_tr = X_train_sc if name == 'SVM' else X_train
    X_te = X_test_sc  if name == 'SVM' else X_test

    fresh_model = model.__class__(**{
        k: v for k, v in model.get_params().items()
        if k in model.__class__().get_params()
    })

    rs = RandomizedSearchCV(
        fresh_model, param_grids[name],
        n_iter=20, cv=cv, scoring='accuracy',
        random_state=RANDOM_STATE, n_jobs=-1, verbose=0
    )

    t0 = time.time()
    rs.fit(X_tr, y_train)
    elapsed = time.time() - t0

    test_acc  = accuracy_score(y_test,  rs.predict(X_te))
    train_acc = accuracy_score(y_train, rs.predict(X_tr))

    random_results[name] = {
        'best_estimator': rs.best_estimator_,
        'best_params'   : rs.best_params_,
        'cv_score'      : rs.best_score_,
        'test_acc'      : test_acc,
        'train_acc'     : train_acc,
        'time'          : elapsed,
        'rs_object'     : rs
    }

    print(f'\n  {name}')
    print(f'    Best CV Accuracy : {rs.best_score_*100:.2f}%')
    print(f'    Test  Accuracy   : {test_acc*100:.2f}%')
    print(f'    Training Time    : {elapsed:.1f}s')
    print(f'    Best Parameters  : {rs.best_params_}')

print('\n' + '=' * 60)
print('Random search complete for all models.')


## 8. Performance Comparison Table

In [ ]:
rows = []
for name in models:
    b  = baseline_results[name]
    gs = grid_results[name]
    rs = random_results[name]
    rows.append({
        'Model'               : name,
        'Default Acc %'       : f"{b['test_acc']*100:.1f}",
        'GridSearch Acc %'    : f"{gs['test_acc']*100:.1f}",
        'RandomSearch Acc %'  : f"{rs['test_acc']*100:.1f}",
        'Improvement (GS) %'  : f"+{(gs['test_acc']-b['test_acc'])*100:.1f}",
        'GS Time (s)'         : f"{gs['time']:.1f}",
        'RS Time (s)'         : f"{rs['time']:.1f}"
    })

comp_df = pd.DataFrame(rows).set_index('Model')
print('\nFull Performance Comparison')
display(comp_df)


## 9. Best Parameters per Model

In [ ]:
for name in models:
    gs = grid_results[name]
    rs = random_results[name]
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print('  GridSearch Best Parameters:')
    for k, v in gs['best_params'].items():
        print(f'    {k} = {v}')
    print('  RandomSearch Best Parameters:')
    for k, v in rs['best_params'].items():
        print(f'    {k} = {v}')


## 10. GridSearch vs RandomSearch Comparison

In [ ]:
rows2 = []
for name in models:
    gs = grid_results[name]
    rs = random_results[name]
    gs_params = ', '.join(f'{k}={v}' for k, v in gs['best_params'].items())
    rs_params = ', '.join(f'{k}={v}' for k, v in rs['best_params'].items())
    rows2.append({'Model': name, 'Method': 'GridSearch',
                  'Test Acc %': f"{gs['test_acc']*100:.2f}",
                  'Time (s)'  : f"{gs['time']:.1f}",
                  'Best Parameters': gs_params})
    rows2.append({'Model': name, 'Method': 'RandomSearch',
                  'Test Acc %': f"{rs['test_acc']*100:.2f}",
                  'Time (s)'  : f"{rs['time']:.1f}",
                  'Best Parameters': rs_params})

cmp2 = pd.DataFrame(rows2).set_index(['Model', 'Method'])
display(cmp2)


## 11. Parameter Impact Visualisations

In [ ]:
def plot_param_impact(gs_object, param_name, model_name, ax):
    results = pd.DataFrame(gs_object.cv_results_)
    col = f'param_{param_name}'
    if col not in results.columns:
        ax.text(0.5, 0.5, f"'{param_name}' not in grid", ha='center', va='center')
        ax.set_title(f'{model_name} -- {param_name}')
        return
    grouped = results.groupby(col)['mean_test_score'].mean().reset_index()
    grouped.columns = [param_name, 'mean_cv_accuracy']
    grouped[param_name] = grouped[param_name].astype(str)
    sns.lineplot(data=grouped, x=param_name, y='mean_cv_accuracy',
                 marker='o', linewidth=2, ax=ax)
    ax.set_title(f'{model_name}: {param_name} vs CV Accuracy')
    ax.set_xlabel(param_name)
    ax.set_ylabel('Mean CV Accuracy')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.grid(True, linestyle='--', alpha=0.6)

tree_models = ['Random Forest', 'XGBoost', 'Gradient Boosting']

# Plot 1: max_depth vs Accuracy
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('max_depth vs CV Accuracy', fontsize=14, fontweight='bold')
for ax, name in zip(axes, tree_models):
    plot_param_impact(grid_results[name]['gs_object'], 'max_depth', name, ax)
plt.tight_layout()
plt.show()

# Plot 2: n_estimators vs Accuracy
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('n_estimators vs CV Accuracy', fontsize=14, fontweight='bold')
for ax, name in zip(axes, tree_models):
    plot_param_impact(grid_results[name]['gs_object'], 'n_estimators', name, ax)
plt.tight_layout()
plt.show()

# Plot 3: learning_rate
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('learning_rate vs CV Accuracy', fontsize=14, fontweight='bold')
for ax, name in zip(axes, ['XGBoost', 'Gradient Boosting']):
    plot_param_impact(grid_results[name]['gs_object'], 'learning_rate', name, ax)
plt.tight_layout()
plt.show()

# Plot 4: SVM C parameter
fig, ax = plt.subplots(figsize=(5, 4))
fig.suptitle('SVM: C vs CV Accuracy', fontsize=14, fontweight='bold')
plot_param_impact(grid_results['SVM']['gs_object'], 'C', 'SVM', ax)
plt.tight_layout()
plt.show()


## 12. Overfitting Analysis

In [ ]:
ov_rows = []
for name in models:
    b  = baseline_results[name]
    gs = grid_results[name]
    ov_rows.append({'Model': name, 'Stage': 'Before Tuning',
                    'Train %': b['train_acc']*100, 'Val %': b['test_acc']*100,
                    'Gap %': (b['train_acc']-b['test_acc'])*100})
    ov_rows.append({'Model': name, 'Stage': 'After Tuning (GS)',
                    'Train %': gs['train_acc']*100, 'Val %': gs['test_acc']*100,
                    'Gap %': (gs['train_acc']-gs['test_acc'])*100})

ov_df = pd.DataFrame(ov_rows)
display(ov_df.set_index(['Model','Stage']).round(2))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Overfitting Analysis: Train vs Validation Accuracy',
             fontsize=14, fontweight='bold')

for ax, name in zip(axes.flat, models.keys()):
    sub = ov_df[ov_df['Model'] == name]
    x   = np.arange(len(sub))
    w   = 0.3
    ax.bar(x - w/2, sub['Train %'], width=w, label='Train',
           color='steelblue', alpha=0.85)
    ax.bar(x + w/2, sub['Val %'],   width=w, label='Val',
           color='darkorange', alpha=0.85)
    ax.set_title(name)
    ax.set_xticks(x)
    ax.set_xticklabels(sub['Stage'], rotation=15, ha='right')
    ax.set_ylabel('Accuracy %')
    ax.set_ylim(50, 105)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    for xi, (tr, va) in enumerate(zip(sub['Train %'], sub['Val %'])):
        ax.text(xi-w/2, tr+0.5, f'{tr:.1f}', ha='center', fontsize=7.5)
        ax.text(xi+w/2, va+0.5, f'{va:.1f}', ha='center', fontsize=7.5)

plt.tight_layout()
plt.show()


## 13. Auto Leaderboard

In [ ]:
lb_rows = []
for name in models:
    best_acc = max(grid_results[name]['test_acc'],
                   random_results[name]['test_acc'])
    lb_rows.append({'Model': name, 'Best Score %': round(best_acc * 100, 2)})

lb_df = (pd.DataFrame(lb_rows)
           .sort_values('Best Score %', ascending=False)
           .reset_index(drop=True))
lb_df.index += 1
lb_df.index.name = 'Rank'
display(lb_df)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['gold', 'silver', '#cd7f32', 'steelblue']
bars = ax.barh(lb_df['Model'][::-1], lb_df['Best Score %'][::-1],
               color=colors[::-1], edgecolor='white', height=0.5)
ax.set_xlabel('Best Accuracy %')
ax.set_title('Model Leaderboard', fontsize=13, fontweight='bold')
ax.set_xlim(lb_df['Best Score %'].min() - 5, 100)
ax.bar_label(bars, fmt='%.2f%%', padding=3)
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 14. Full Results Summary Chart

In [ ]:
method_data = {name: {
    'Default'     : baseline_results[name]['test_acc'] * 100,
    'GridSearch'  : grid_results[name]['test_acc']     * 100,
    'RandomSearch': random_results[name]['test_acc']   * 100,
} for name in models}

summary_df = pd.DataFrame(method_data).T
x     = np.arange(len(summary_df))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width, summary_df['Default'],      width, label='Default',
       color='#4C72B0', alpha=0.9)
ax.bar(x,         summary_df['GridSearch'],   width, label='GridSearch',
       color='#55A868', alpha=0.9)
ax.bar(x + width, summary_df['RandomSearch'], width, label='RandomSearch',
       color='#C44E52', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(summary_df.index, rotation=15, ha='right')
ax.set_ylabel('Test Accuracy %')
ax.set_title('Default vs GridSearch vs RandomSearch -- All Models',
             fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylim(50, 102)
ax.grid(axis='y', linestyle='--', alpha=0.4)
for i, (d, g, r) in enumerate(zip(summary_df['Default'],
                                   summary_df['GridSearch'],
                                   summary_df['RandomSearch'])):
    ax.text(i - width, d + 0.3, f'{d:.1f}', ha='center', fontsize=7.5)
    ax.text(i,         g + 0.3, f'{g:.1f}', ha='center', fontsize=7.5)
    ax.text(i + width, r + 0.3, f'{r:.1f}', ha='center', fontsize=7.5)
plt.tight_layout()
plt.show()


## 15. Final Recommendation Report

In [ ]:
# Find best model overall
best_name, best_acc, best_src = None, 0, None
for name in models:
    for src_name, res in [('GridSearch', grid_results[name]),
                           ('RandomSearch', random_results[name])]:
        if res['test_acc'] > best_acc:
            best_acc  = res['test_acc']
            best_name = name
            best_src  = src_name
            best_res  = res

default_acc = baseline_results[best_name]['test_acc']
improvement = (best_acc - default_acc) * 100

gs_acc = grid_results[best_name]['test_acc']   * 100
rs_acc = random_results[best_name]['test_acc'] * 100
gs_t   = grid_results[best_name]['time']
rs_t   = random_results[best_name]['time']

train_sc = best_res['train_acc'] * 100
val_sc   = best_res['test_acc']  * 100
gap      = train_sc - val_sc
gap_str  = 'acceptable' if gap < 10 else 'possible overfitting'

bp_str = '\n'.join(f'  - {k} = {v}' for k, v in best_res['best_params'].items())

print('\n' + '=' * 62)
print('       === FINAL RECOMMENDATION REPORT ===')
print('=' * 62)
print(f'\nBest Performing Model : {best_name}  (via {best_src})')
print('\nPerformance Improvement:')
print(f'  Default Accuracy : {default_acc*100:.1f}%')
print(f'  Tuned Accuracy   : {best_acc*100:.1f}%')
print(f'  Improvement      : +{improvement:.1f}%')
print(f'\nBest Parameters:\n{bp_str}')
print(f'\nGridSearch vs RandomSearch (for {best_name}):')
print(f'  GridSearch   -> {gs_acc:.2f}% in {gs_t:.1f}s')
print(f'  RandomSearch -> {rs_acc:.2f}% in {rs_t:.1f}s')
print('  Recommendation: GridSearch for max accuracy; RandomSearch for speed.')
print('\nOverfitting Check:')
print(f'  Training Score   : {train_sc:.1f}%')
print(f'  Validation Score : {val_sc:.1f}%')
print(f'  Gap              : {gap:.1f}%  ({gap_str})')
print('\n' + '=' * 62)
print('  LEADERBOARD (Best Tuned Accuracy)')
print('=' * 62)
for rank, name in enumerate(
    sorted(models, key=lambda n: max(
        grid_results[n]['test_acc'], random_results[n]['test_acc']
    ), reverse=True), 1
):
    acc  = max(grid_results[name]['test_acc'],
               random_results[name]['test_acc']) * 100
    star = ' <-- WINNER' if name == best_name else ''
    print(f'  {rank}. {name:<22} {acc:.2f}%{star}')
print('=' * 62)
